In [ ]:
import kwant
import matplotlib.pyplot as plt
import numpy as np
from typing import Callable
from typing import Tuple
from typing import List
import tinyarray as ta
import types as tp
import warnings
import copy 
from matplotlib.ticker import ScalarFormatter
from scipy import linalg as la

def eV2au(energy): #eV -> a.u.
    return energy*0.03674932587122423
def nm2au(length): #nm -> a.u.
    return length*18.89726133921252

def delta_Kr(i: int, j: int) -> float: # Kronecker delta
    return 1.0 if i == j else 0.0

# jednoorbitalowe macierze Pauliego
sigma_x_1orb = ta.array([[0, 1], [1, 0]])
sigma_y_1orb = ta.array([[0, -1j], [1j, 0]])
sigma_z_1orb = ta.array([[1, 0], [0, -1]])
sigma_z_up = ta.array([[1, 0], [0, 0]])
sigma_z_down = ta.array([[0, 0], [0, 1]])

# macierze Pauliego dla ukladu
s_0 = np.identity(6)
s_x = np.kron(sigma_x_1orb, np.identity(3))
s_y = np.kron(sigma_y_1orb, np.identity(3))
s_z = np.kron(sigma_z_1orb, np.identity(3))
s_z_up = np.kron(sigma_z_up, np.identity(3))
s_z_down = np.kron(sigma_z_down, np.identity(3))

def Pauli_matrices_0xyz(N_modes: int) -> Tuple[np.ndarray]:
    zero = np.kron(np.identity(N_modes), s_0)
    x = np.kron(np.identity(N_modes), s_x)
    y = np.kron(np.identity(N_modes), s_y)
    z = np.kron(np.identity(N_modes), s_z)
    up = np.kron(np.identity(N_modes), s_z_up)
    down = np.kron(np.identity(N_modes), s_z_down)
    return zero, x, y, z, up, down

SpinOrbit_matrix = np.array(([0, 1j, 0, 0, 0, -1],
                            [-1j, 0, 0, 0, 0, 1j],
                            [0, 0, 0, 1, -1j, 0],
                            [0, 0, 1, 0, -1j, 0],
                            [0, 0, 1j, 1j, 0, 0],
                            [-1, -1j, 0, 0, 0, 0]))

def pm_mod(pm, **kwargs):
    pm_lokalne = copy.copy(pm)
    for variable, value in kwargs.items():
        if value is not None:
            setattr(pm_lokalne, variable, value)
    return pm_lokalne

def draw(x: np.ndarray, y_list: list,\
        axis_x: str = 'x', axis_y: str = 'y', y_range: list = [None, None],\
        color_list: list = [None]) -> None:
    '''
    draw a nd.array object (made for disperssion)

    x - x-axis values, y_list - list of np.ndarray type y-axis value arrays,
    axis_x - x-axis description, axis_y - y-axis description,
    y_range - range of y-axis (as [y_min, y_max]),
    color_list - list of colors

    '''
    
    if not(isinstance(axis_x, str)) or not(isinstance(axis_x, str)):
        return 0
    else:
        plt.figure(figsize=(10,7))
        if len(color_list) != 1:
            for i, y in enumerate(y_list):
                plt.plot(x,y, color = color_list[i])
        else:
            for i, y in enumerate(y_list):
                plt.plot(x,y, color = color_list[0])
        x_min, x_max = np.min(x), np.max(x)
        xticks = np.linspace(x_min, x_max, 5)
        plt.xticks(xticks, [f"{tick:.2f}" for tick in xticks])
        #plt.style.use('seaborn-paper')
        plt.xlabel(axis_x,fontsize=30); plt.ylabel(axis_y,fontsize=30)
        plt.tick_params(axis='both', which='major', labelsize=30)
        if y_range != [None, None]:
            plt.ylim(y_range[0], y_range[1])
        plt.show()

# SKYRMION

In [ ]:
class Skyrmion:
    '''
    class defining a single skyrmion
    __init__

    x0, y0: skyrmion position,
    r: skyrmion radius,

    m: skyrmion helicity ("set to 1 for a single skyrmion with,
    m_z = 1 far from the center and -1 at the center"),

    gamma: skyrmion vorticity... 0 (Neel) -> pi/2 (Bloch),
    
    name: give your skyrmion a name
    '''

    def __init__(self,*, x0: float, y0: float, r: float,
                m: float, gamma: float, name: str = None):
        self.x0 = x0; self.y0 = y0; self.r = r; self.m = m;
        self.gamma = gamma; self.name = name
    
    def __call__(self, n_rad: float, n_pts: int) -> None:
        '''
        plot your skyrmion (lattice: n_pts x n_pts)
        plotted area: n_rad its radius
        '''
        linspx = np.linspace(-n_rad*self.r + self.x0, n_rad*self.r + self.x0, n_pts)
        linspy = np.linspace(-n_rad*self.r + self.y0, n_rad*self.r + self.y0, n_pts)
        X, Y = np.meshgrid(linspx, linspy)
        s_i = [(i, j) for i in range(n_pts) for j in range(n_pts)]
        s = [(x, y) for x in list(linspx) for y in list(linspy)]
        sites = [tp.SimpleNamespace(id = s_i[idx], pos = s[idx])\
                for idx in range(len(s))]

        m = np.zeros((n_pts,n_pts,3))
        for site in sites:
            (i,j) = site.id
            m[i,j,:] = self.skyrmion(site)
        m_x = np.zeros((n_pts,3))
        for idx, x in enumerate(linspx):
            m_x[idx,:] = self.skyrmion(tp.SimpleNamespace(pos = (x, self.y0)))

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        colors = {'x': 'r', 'y': 'b', 'z': 'g'}
        for id, nam in enumerate(['x', 'y', 'z']):
            contour = axes[0,id].contourf(Y, X, m[:, :, id], cmap='Greys', levels = 200, alpha=0.7)
            axes[0,id].set_xlabel('y/dx'); axes[0,id].set_ylabel('x/dx')
            axes[0,id].set_aspect('equal')
            fig.colorbar(contour, ax=axes[0,id], label=f"$m_{nam}$")
            axes[0,id].set_title(f"$m_{nam}$")

            contour = axes[1,0].plot(m_x[:, id], label = f"$m_{nam}$", color = colors[nam], linewidth=2)
            axes[1,0].set_xlabel("x"); axes[1, 0].legend(loc='lower right', fontsize=15)
            axes[1,0].set_ylabel('m'), axes[1, 0].set_ylim(-1.2, 1.2); axes[1,0].set_title(f"$m(x,y=center)$")

        fig.suptitle(f"{self.name}", fontsize = 25); plt.tight_layout(); plt.show()

    def _coordinates(self, site: kwant.builder.Site) -> Tuple[float, float, float]:

        (x,y) = site.pos
        r = np.sqrt((x - self.x0)**2 + (y - self.y0)**2)

        if r <= self.r:
            theta_r = np.arccos((2.0*r-self.r)/self.r) # pi -> 0
        elif r > self.r:
            theta_r = 2.0*np.pi - 4.0*np.arctan(np.exp(4*r/self.r))

        phi_xy = np.arctan2(y - self.y0, x - self.x0)

        return (r, phi_xy, theta_r)
    
    def skyrmion(self, site: kwant.builder.Site) -> np.ndarray:
        '''
        defines localized magnetic
        moment of a skyrmion
        '''
        
        r, phi_xy, theta_r = self._coordinates(site)
        S_x = -np.cos(self.m*phi_xy + self.gamma)*np.sin(theta_r)
        S_y = -np.sin(self.m*phi_xy + self.gamma)*np.sin(theta_r)
        S_z = np.cos(theta_r)

        return np.array([S_x, S_y, S_z])

# MATRIX ELEMENTS

In [ ]:
def get_table_of_integrals(pm: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    calculates matrix elements for the "z" direction 
    of 2DEG confinement electric potential,
    momentum and momentum squared
    '''

    nz=int(pm['D']/pm['dz'])+1
    
    mV = np.zeros((pm['N_modes'],pm['N_modes']), dtype=np.complex_)
    mkz = np.zeros((pm['N_modes'],pm['N_modes']), dtype=np.complex_)
    mk2z = np.zeros((pm['N_modes'],pm['N_modes']), dtype=np.complex_)
    
    df = np.zeros((nz,pm['N_modes']), dtype=np.complex_)
    d2f = np.zeros((nz,pm['N_modes']), dtype=np.complex_)
    
    position=np.zeros(nz)
    potential=np.zeros(nz)
    H = np.zeros((nz-2,nz-2),dtype=np.complex_)
    psi = np.zeros((nz,pm['N_modes']),dtype=np.complex_)

    for i in range(nz-2):
        z= (i+1)*pm['dz']
        position[i+1]=z
        potential[i+1]=pm['F']*z
        H[i, i] = pm['D']*2/pm['dz']**2 + pm['F']*z
        if i > 0:
            H[i, i-1] = H[i-1, i] = -pm['D']*1.0/pm['dz']**2 
    position[nz-1]=z+pm['dz']
    potential[nz-1]=pm['F']*(z+pm['dz'])
    
    #diagonalizacja
    val, vec = la.eigh(H)

    for i in range(nz-2):
        for j in range(pm['N_modes']):
            psi[i+1,j]=vec[i,j]
        
    #normalizacja
    for i in range(pm['N_modes']):
        f = psi[:,i]
        c = np.sqrt(np.sum(abs(f)**2*pm['dz']))
        psi[:,i]=psi[:,i]/c
        f=f/c

        #pochodne 1st and 2nd
        for j in range(1,nz-1):
            df[j,i]=(f[j+1]-f[j-1])/(2*pm['dz'])
        df[0,i]=(f[1]-f[0])/pm['dz']
        df[-1,i]=(f[-1]-f[-2])/pm['dz']

        for j in range(1,nz-1):
            d2f[j,i]=(f[j-1]-2*f[j]+f[j+1])/(pm['dz']**2)
        d2f[0,i]=(df[1,i]-df[0,i])/pm['dz']
        d2f[-1,i]=(df[-1,i]-df[-2,i])/pm['dz']

    ### liczenie calek
    def integral(f, n, dx):
        ret = 0.0
        for k in range(n):
            ret += f[k] * dx
        return ret

    for i in range(pm['N_modes']):
        for j in range(pm['N_modes']):
            ff=psi[:,i]
            ff1=psi[:,j]
            dff=df[:,j]
            d2ff=d2f[:,j]
                    
            fxV = ff*potential*ff1    
            fxdf = ff * dff
            fxd2f = ff * d2ff

            calka_V = integral(fxV,nz,pm['dz'])
            calka_df = integral(fxdf,nz,pm['dz'])
            calka_d2f = integral(fxd2f,nz,pm['dz'])
            
            mV[i,j]=calka_V
            mkz[i,j]=-1j*calka_df
            mk2z[i,j]=-calka_d2f
            
    return mV, mkz, mk2z

# LOCALIZED $\mu$ + MAKESYSTEM 

In [ ]:
def rectangle_maker(pm: dict) -> Tuple[Callable[[tuple],bool]]:
    '''
    defines a shape of the system
    and the hall-bar geometry leads
    '''

    def rectangle(pos: tuple) -> bool:
        (x,y) = pos
        return (0 <= x <= pm['L']) and (-pm['W']/2 <= y <= pm['W']/2)
    
    def lead_h(pos: tuple) -> bool:
        (x,y) = pos
        return (-pm['W']/2 <= y <= pm['W']/2)
    
    def lead_v(pos: tuple) -> bool:
        (x,y) = pos
        return (-pm['W_hb']/2 <= x - pm['L']/2 <= pm['W_hb']/2)
    
    return (rectangle, lead_h, lead_v)

def magnetic_moment_maker(pm: dict) -> Callable[[kwant.builder.Site], np.ndarray]:
    '''
    creates a function that
    defines a local magnetic moment
    in a presence of multiple skyrmions
    '''

    def magnetic_moment(site: kwant.builder.Site) -> np.ndarray:
        skyrmions: list = pm['skyrmions']
        (x,y) = site.pos
        mm = np.array([0.0, 0.0, 0.0])
        for skyrmion in skyrmions:
            new = np.all(mm == np.array([0.0, 0.0, 0.0]))
            if np.sqrt((x-skyrmion.x0)**2 + (y-skyrmion.y0)**2) <= 1.5*skyrmion.r:
                if not new:
                    warnings.warn("Skyrmions overlaped.", UserWarning)
                mm = skyrmion.skyrmion(site)
            else:
                pass
        if np.all(mm == np.array([0.0, 0.0, 0.0])):
            mm = np.array([0.0, 0.0, 1.0])

        return mm
    return magnetic_moment

def makesystem(pm: dict, plot_S_z: bool = False) -> kwant.builder.FiniteSystem:
    '''
    provide system parameters as a dictionary:

    'dx' - lattice constant,
    'L' - system length,
    'W' - system width,
    'm_eff' - effective mass,

    'dz' - perp. lattice constant,
    'D' - 2DEG thickness,
    'N_modes' - number of transverse modes in "z" direction,
    'F' - 2DEG confinement electric field [a.u.]

    'W_hb' - width of a hall-bar vertical contact,
    'skyrmions' - list of Skyrmion objects,

    'dft_L' - effective mass "L" parameter
    'dft_M' - effective mass "M" parameter
    'dft_N' - effective mass "N" parameter
    'dft_Delta_T' - temperature parameter
    'dft_SpinOrbit' - spin-orbit coupling strength
    'J_sd' - magnetic moment coupling strength

    ###########################################
    you can plot skyrmions' magnetic moment s_z
    setting plot_S_z = True
    ###########################################

    '''
    s_zero, s_x, s_y, s_z, s_up, s_down = Pauli_matrices_0xyz(pm['N_modes'])
    V_el, kz_el, k2z_el = get_table_of_integrals(pm)
    S = magnetic_moment_maker(pm)

    # ------------------------ onsite en.------------------------

    hamiltonian_onsite = np.zeros((pm['N_modes']*6,pm['N_modes']*6), dtype = np.complex_)
    for i in range(pm['N_modes']):
        for j in range(pm['N_modes']):
            val1 = pm['dft_M']*k2z_el[i,j] + V_el[i,j] + delta_Kr(i,j)*2*(pm['dft_L'] + pm['dft_M'])/pm['dx']**2
            val2 = pm['dft_L']*k2z_el[i,j] + V_el[i,j] +  delta_Kr(i,j)*(4*pm['dft_M']/pm['dx']**2 + pm['dft_Delta_T'])
            soc = delta_Kr(i,j)*pm['dft_SpinOrbit']*SpinOrbit_matrix/3
            hamiltonian_onsite[6*i:6*(i+1), 6*j:6*(j+1)] = np.kron(np.identity(2), np.diag([val1, val1, val2])) + soc
            
    def onsite(site):

        S_x, S_y, S_z = S(site)
        (x,y) = site.pos
        
        return ta.array(hamiltonian_onsite - pm['J_sd']*(S_x*s_x + S_y*s_y + S_z*s_z))

    # ------------------------ hopping els.------------------------
    
    # (1,0)
    hamiltonian_10_3x3b = np.array([[-pm['dft_L']/pm['dx']**2, 0, 0],
                                    [0, -pm['dft_M']/pm['dx']**2, 0],
                                    [0, 0, -pm['dft_M']/pm['dx']**2]])
    hamiltonian_10b = ta.array(np.kron(np.identity(pm['N_modes']*2), hamiltonian_10_3x3b))
    
    hamiltonian_10a = np.zeros((pm['N_modes']*6,pm['N_modes']*6), dtype = np.complex_)
    for i in range(pm['N_modes']):
        for j in range(pm['N_modes']):
            
            hamiltonian_10_3x3a = np.array([[0, 0, -0.5j*pm['dft_N']*kz_el[i,j]/pm['dx']],
                                            [0, 0, 0],
                                            [-0.5j*pm['dft_N']*kz_el[i,j]/pm['dx'], 0, 0]])
            
            hamiltonian_10a[6*i:6*(i+1), 6*j:6*(j+1)] = ta.array(np.kron(np.identity(2), hamiltonian_10_3x3a))

    def hopping_10(site1, site2):
        return hamiltonian_10a + hamiltonian_10b 
    
    # (0,1)
    hamiltonian_01_3x3b  = np.array([[-pm['dft_M']/pm['dx']**2, 0, 0],
                                    [0, -pm['dft_L']/pm['dx']**2, 0],
                                    [0, 0, -pm['dft_M']/pm['dx']**2]])
    hamiltonian_01b = ta.array(np.kron(np.identity(pm['N_modes']*2), hamiltonian_01_3x3b))

    hamiltonian_01a = np.zeros((pm['N_modes']*6,pm['N_modes']*6), dtype = np.complex_)
    for i in range(pm['N_modes']):
        for j in range(pm['N_modes']):
            hamiltonian_01_3x3a  = np.array([[0, 0, 0],
                                            [0, 0, -0.5j*pm['dft_N']*kz_el[i,j]/pm['dx']],
                                            [0, -0.5j*pm['dft_N']*kz_el[i,j]/pm['dx'], 0]])
            
            hamiltonian_01a[6*i:6*(i+1), 6*j:6*(j+1)] = ta.array(np.kron(np.identity(2), hamiltonian_01_3x3a))

    def hopping_01(site1, site2):
        return hamiltonian_01a + hamiltonian_01b
    
    # (1,-1)
    
    hamiltonian_1m1_3x3  = np.array([[0, 0.25*pm['dft_N']/pm['dx']**2, 0],
                                    [0.25*pm['dft_N']/pm['dx']**2, 0, 0],
                                    [0, 0, 0]])
    
    hamiltonian_1m1 = ta.array(np.kron(np.identity(pm['N_modes']*2), hamiltonian_1m1_3x3))

    def hopping_1m1(site1, site2):
        return hamiltonian_1m1
    
    # (1,1)

    hamiltonian_11_3x3  = np.array([[0, -0.25*pm['dft_N']/pm['dx']**2, 0],
                                    [-0.25*pm['dft_N']/pm['dx']**2, 0, 0],
                                    [0, 0, 0]])
    
    hamiltonian_11 = ta.array(np.kron(np.identity(pm['N_modes']*2), hamiltonian_11_3x3))
    
    def hopping_11(site1, site2):
        return hamiltonian_11
    
    lat = kwant.lattice.square(pm['dx'], norbs = pm['N_modes']*6)
    shape, lead_h_shape, lead_v_shape = rectangle_maker(pm)

    system = kwant.Builder()
    system[lat.shape(shape, (0, 0))] = onsite
    system[(kwant.builder.HoppingKind((1,0), lat, lat))] = hopping_10
    system[(kwant.builder.HoppingKind((0,1), lat, lat))] = hopping_01
    system[(kwant.builder.HoppingKind((1,-1), lat, lat))] = hopping_1m1
    system[(kwant.builder.HoppingKind((1,1), lat, lat))] = hopping_11
    
    lead_h = kwant.Builder(kwant.TranslationalSymmetry((-pm['dx'],0)))
    lead_h[lat.shape(lead_h_shape, (0, 0))] = onsite
    lead_h[(kwant.builder.HoppingKind((1,0), lat, lat))] = hopping_10
    lead_h[(kwant.builder.HoppingKind((0,1), lat, lat))] = hopping_01
    lead_h[(kwant.builder.HoppingKind((1,-1), lat, lat))] = hopping_1m1
    lead_h[(kwant.builder.HoppingKind((1,1), lat, lat))] = hopping_11
    system.attach_lead(lead_h); system.attach_lead(lead_h.reversed())

    lead_v = kwant.Builder(kwant.TranslationalSymmetry((0,-pm['dx'])))
    lead_v[lat.shape(lead_v_shape, (pm['L']/2, 0))] = onsite
    lead_v[(kwant.builder.HoppingKind((1,0), lat, lat))] = hopping_10
    lead_v[(kwant.builder.HoppingKind((0,1), lat, lat))] = hopping_01
    lead_v[(kwant.builder.HoppingKind((1,-1), lat, lat))] = hopping_1m1
    lead_v[(kwant.builder.HoppingKind((1,1), lat, lat))] = hopping_11
    system.attach_lead(lead_v); system.attach_lead(lead_v.reversed())

    if plot_S_z == True:
        kwant.plotter.map(system, lambda site: S(site)[2], vmin=-1,vmax=1);
    system = system.finalized()

    return system

In [ ]:
#gauge = kwant.physics.magnetic_gauge(system)

# CALCULATIONS

In [ ]:
def disperssion(system: kwant.builder.FiniteSystem,\
                pm: dict, nr_lead: int,\
                k_max: float, n_k_steps: int) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Calculate disperssion relation in a lead (nr_lead).
    Returns wave-vectors and corresponding energies.
    '''

    bands = kwant.physics.Bands(system.leads[nr_lead])

    ks = np.linspace(-k_max*pm['dx'], k_max*pm['dx'], n_k_steps)
    Es = [bands(k) for k in ks]

    return (ks/pm['dx'], np.asarray(Es))

def plotter(system: kwant.builder.FiniteSystem, to_plot: np.ndarray, d_or_c: str) -> None:
    
    cmap = {'i2': 'Grays', 's_u': 'Reds', 's_d': 'Blues',
            's_x': 'coolwarm', 's_y': 'coolwarm', 's_z': 'coolwarm'}
    
    pos = {'i2': (0,0), 's_u': (0,1), 's_d': (0,2),
            's_x': (1,0), 's_y': (1,1), 's_z': (1,2)}
    
    kgs = {'fig_size': (6,6), 'relwidth': 0.05, 'show': False, 'colorbar': False, 'syst': system}
    if d_or_c == 'd':
        plot = lambda to_plot, cmap, ax: kwant.plotter.density(density = to_plot, ax = ax, cmap = cmap, **kgs)
        names = {'i2': r'$|\Psi|^2$', 's_u': r'$|\Psi_\uparrow|^2$', 's_d': r'$|\Psi_\downarrow|^2$',
            's_x': r'$\Psi^*\sigma_x \Psi$', 's_y': r'$\Psi^*\sigma_y \Psi$', 's_z': r'$\Psi^*\sigma_z \Psi$'}
    elif d_or_c == 'c':
        plot = lambda to_plot, cmap, ax: kwant.plotter.current(current = to_plot, ax = ax, cmap = cmap, **kgs)
        names = {'i2': r'$J$', 's_u': r'$J_\uparrow$', 's_d': r'$J_\downarrow$',
            's_x': r'$J_x$', 's_y': r'$J_y$', 's_z': r'$J_z$'}
    else:
        warnings.warn('Specify whether you want plot densities or currents.'); return 0
    
    fig, axes = plt.subplots(2, 3, figsize=(36, 18))
    cbars, im = {}, {}
    formatter = ScalarFormatter(useMathText=True)
    for key, val in pos.items():
        im[key] = plot(to_plot=to_plot[key], cmap=cmap[key], ax = axes[val[0],val[1]])
        cbars[key] = fig.colorbar(im[key], ax=axes[val[0],val[1]])
        cbars[key].formatter = formatter; cbars[key].update_ticks()
        cbars[key].ax.tick_params(labelsize=15); cbars[key].ax.yaxis.offsetText.set_fontsize(15);
        axes[val[0],val[1]].set_title(names[key], fontsize=20)

    plt.show()

def spin_analysis(system: kwant.builder.FiniteSystem, Energy_au: float, n_orbitals: int) -> None:
    '''
    Plots densities and currents of some operators for a defined system.

    Energy_au - energy of an electron in atomic units (a.u.),
    n_orbitals - number of transverse modes in "z" direction
    '''
    
    def sum_over_modes(operator: kwant.operator.Density | kwant.operator.Current,\
                        psi: np.ndarray) -> np.ndarray:
        
        val_total = operator(psi[0])
        for mode in range(1,len(psi)):
            val_mode = psi[mode]
            val_total += operator(val_mode)
        
        return val_total
    
    idn, s_x, s_y, s_z, s_u, s_d = Pauli_matrices_0xyz(n_orbitals)
    op = {'s_x': s_x, 's_y': s_y, 's_z': s_z,
            's_u': s_u, 's_d': s_d,'i2': idn}

    psi = kwant.wave_function(sys=system, energy=Energy_au)(0)

    density, current, source  = {}, {}, {}
    for key, operator in op.items():
        density[key] = sum_over_modes(kwant.operator.Density(system, operator),psi)
        current[key] = sum_over_modes(kwant.operator.Current(system, operator),psi)
        #source[key] = sum_over_modes(kwant.operator.Source(system, 1j*operator,\
        #                                                    check_hermiticity=False),psi)
        
    plotter(system, density,'d')
    plotter(system, current,'c')

    return density, current, source

In [ ]:
def Hall_conductance(system: kwant.builder.FiniteSystem, Energies_au: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    ''' Calculate Hall-conductance of a system '''

    Energies_eV = np.asarray(Energies_au)/eV2au(1.0)
    hallG = []
    for idx, E_au in enumerate(Energies_au):
        S = kwant.smatrix(system, E_au, in_leads = (0,1,2,3), out_leads = (0,1,2,3))
        s_in_ = S.transmission(1,2)-S.transmission(2,1)
        hallG.append(s_in_)

    return (Energies_eV, np.asarray(hallG))

def Hall_conductance_moving_skyrmion(pm: dict, skyrmions_pos_stop: List[Tuple[float,float]], Energy_au: float, num: int) -> Tuple[np.ndarray, np.ndarray]:
    ''' Calculate Hall-conductance of a system while moving\\
    a skyrmion with an energy value that is set,
    '''
    hallG_moving = []

    pm_c = copy.deepcopy(pm)
    positions = []
    for idx, skyrmion in enumerate(pm_c['skyrmions']):
        pos_s = (skyrmion.x0, skyrmion.y0)
        pos = skyrmions_pos_stop[idx]
        x = np.linspace(start = pos_s[0], stop = pos[0], num = num)
        y = np.linspace(start = pos_s[1], stop = pos[1], num = num)
        positions.append((x,y))

    def _skymion_pos_mod(skyrmion: Skyrmion, x0: float, y0: float) -> Skyrmion:
        skyrmion.x0 = x0
        skyrmion.y0 = y0

        return skyrmion

    for idx in range(num):
        
        skyrmions_local = [_skymion_pos_mod(skyr, x0=positions[idxx][0][idx], y0=positions[idxx][1][idx]) for idxx, skyr in enumerate(pm_c['skyrmions'])]

        pm_c['skyrmions'] = skyrmions_local
        system_local = makesystem(pm_c)
        Energies_eV, hallG_moved = Hall_conductance(system_local, Energies_au=[Energy_au])
        hallG_moving.append(hallG_moved)

    return positions, hallG_moving

# EXAMPLE

In [ ]:
skyrmion1 = Skyrmion(x0 = nm2au(100.0),
                    y0 = nm2au(0.0),
                    r = nm2au(16.0),
                    gamma = 0.0,
                    m = 1.0,
                    name = 'mój skyrmion'
                    )

skyrmion2 = Skyrmion(x0 = nm2au(70),
                    y0 = nm2au(0),
                    r = nm2au(7),
                    gamma = 0.0,
                    m = 1.0,
                    name = 'mój drugi skyrmion'
                    )

#skyrmion1(n_rad = 2, n_pts = 51)
#skyrmion2(n_rad = 2, n_pts = 51)

pm = {
    'dx': nm2au(1.0), # lattice constant
    'L': nm2au(297.0), # system length
    'W': nm2au(33.0), # system width
    'm_eff': 0.016, # effective mass

    'dz': nm2au(1.0), # perp. lattice constant
    'D': nm2au(60.0), # 2DEG thickness
    'N_modes': 2, # number of transverse modes in "z" direction
    'F': 3.0*eV2au(1e-3)/nm2au(1.0), # 2DEG confinement electric field

    'dft_L': 0.6104 *eV2au(1)*(nm2au(0.1))**2, # effective mass "L" parameter
    'dft_M': 9.73 *eV2au(1)*(nm2au(0.1))**2, # effective mass "M" parameter
    'dft_N': -1.616 *eV2au(1)*(nm2au(0.1))**2, # effective mass "N" parameter
    'dft_Delta_T': 2.1 *eV2au(1e-3), # temperature parameter
    'dft_SpinOrbit': 28.5 *eV2au(1e-3), # spin-orbit coupling strength
    'J_sd': 28.5 *eV2au(1e-2), # magnetic moment coupling strength --- !!! DO POPRAWY !!!

    'W_hb': nm2au(33.0), #width of a hall-bar vertical contact
    'skyrmions': [skyrmion1] # list of skyrmion type objects
}

In [ ]:
system = makesystem(pm, plot_S_z=True)
kwant.plot(system);

ks, Es = disperssion(system, pm = pm, nr_lead=0,\
                    k_max = 0.5/nm2au(1), n_k_steps= 40)
print(f'minimum pasm lead: {np.round(np.min(Es/eV2au(1.0)), 4)} meV')
draw(ks*nm2au(1), [Es/eV2au(1)], r'$k_x$ (1/nm)', r'$E$ (eV)', y_range=[-0.238,-0.233])

In [ ]:
d, c, s = spin_analysis(system, eV2au(-0.236e-3), n_orbitals= 2)

In [ ]:
positions, hallG_moving = Hall_conductance_moving_skyrmion(pm = pm, skyrmions_pos_stop = [(nm2au(200.0),0.0)], Energy_au = eV2au(-.8), num = 60)
plt.plot(hallG_moving)